In [1]:
import pandas as pd
import numpy as np

from tableone import TableOne
from tabulate import tabulate, SEPARATING_LINE


In [2]:
train_panel_in = "../../../data/final/train_panel.parquet"
bike_panel_in = "../../../data/final/bike_panel.parquet"
uber_panel_in = "../../../data/final/uber_panel.parquet"

balance_out = "../../../reports/replication/balance.md"

In [3]:
model_panel = pd.concat([pd.read_parquet(train_panel_in),
                         pd.read_parquet(bike_panel_in),
                         pd.read_parquet(uber_panel_in)],
                        keys=['train','bike','uber'], names=['transit']).reset_index('transit')

# Balance

In [5]:
def fix_count_rows(tone, count_row):
    row_labels = tone.tableone.index.get_level_values(0)
    is_count = row_labels.str.contains(count_row)
    new_labels =  np.where(is_count,
                            row_labels.str.replace(', mean (SD)',''),
                            row_labels)
    tone.tableone.index = pd.MultiIndex.from_tuples(zip(new_labels, 
                                            tone.tableone.index.get_level_values(1).values))
    return tone

def balance_table(df, unit_name, unit_abbr):
    space_data = df.drop_duplicates(['transit','id']).copy()
    space_table = TableOne(space_data,
         categorical=['transit'],
         groupby='UCMP',
         rename={'UCMP':'Near DNC', 'n':unit_name},
         pval=True,
         missing=False,
         overall=False)
    
    space_table = fix_count_rows(space_table, unit_name)

    time_data = df.drop_duplicates(['transit','id','UCMP','date','rides']).copy()
    time_table = TableOne(time_data,
                      columns=['rides','logrides'],
                      groupby='UCMP',
                      rename={'UCMP':'Near DNC',
                              'n':f'{unit_abbr}-days',
                              'rides': 'daily rides',
                              'logrides': 'log(daily rides)'},
                      pval=True,
                      missing=False,
                      overall=False)
    
    time_table = fix_count_rows(time_table, unit_abbr)

    
    def apply_order(x, keys):
        orders = np.zeros_like(x)
        for i, key in enumerate(keys):
            orders += i * x.str.contains(key)
        return orders

    key_order = [unit_name, unit_abbr, 'rides', 'transit']
    order_func = lambda x: apply_order(x, key_order)
    bal_table = pd.concat([space_table.tableone, time_table.tableone])
    bal_table = bal_table.sort_index(key=order_func)
    
    # Hack to concat TableOne tables.
    space_table.tableone = bal_table
    return space_table

In [ ]:
balance_table(df.loc[df.transit==transit], unit_name+'s', unit_name)

## Nominal Model

In [6]:
def balance_transit(df, transit, unit_name):
    tbl = balance_table(df.loc[df.transit==transit], unit_name+'s', unit_name)
    tbl.tableone = tbl.tableone[~tbl.tableone.index.get_level_values(0).str.contains('transit')]
    tbl.tableone.index = tbl.tableone.index.set_levels(levels=[transit],level=1)
    return tbl

In [7]:
train_balance = balance_transit(model_panel, 'train', 'station').tableone.loc[lambda x: ~x.index.get_level_values(0).str.contains('train')]
bike_balance = balance_transit(model_panel, 'bike', 'dock').tableone.loc[lambda x: ~x.index.get_level_values(0).str.contains('bike')]
uber_balance = balance_transit(model_panel, 'uber', 'tract').tableone
tbl = pd.concat([train_balance, bike_balance, uber_balance])
tbl.columns = tbl.columns.droplevel(0)
tbl = tbl.swaplevel(axis=0)
tbl = tbl.reset_index()
tbl['level_0'] = tbl['level_0'].where(~tbl['level_0'].duplicated(),'')
tabulate(tbl, headers=['','','Not Near DNC','Near DNC','P-Value'],tablefmt='html',showindex=False)

,,Not Near DNC,Near DNC,P-Value
train,stations,134,8,
,station-days,11256,637,
,"daily rides, mean (SD)",2250.5 (2195.3),1398.6 (1121.1),<0.001
,"log(daily rides), mean (SD)",7.3 (0.9),6.7 (1.7),<0.001
,"bus_distance, mean (SD)",0.1 (0.3),0.0 (0.0),0.010
,"bike_distance, mean (SD)",0.2 (0.5),0.1 (0.1),0.002
,"sqrt(area), mean (SD)",1.5 (2.2),1.9 (2.2),0.578
,"lat, mean (SD)",0.0 (0.0),0.0 (0.0),<0.001
,"long, mean (SD)",-0.1 (0.0),-0.1 (0.0),0.040
bike,docks,1519,47,


In [8]:
tbl_data = []
for i,row in enumerate(tbl.to_numpy()):
    if row[0] != '' and i != 0:
        tbl_data.append(SEPARATING_LINE)
    tbl_data.append(row.tolist())

In [9]:
with open(balance_out,"w") as f:
    f.write(tabulate(tbl_data, headers=['','','Not Near DNC','Near DNC','P-Value'],tablefmt='github',showindex=False))
!cp {balance_out} ../../../../eric-mc2-cv/static/uploads/